In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import json

print("networkx version :", nx.__version__)

In [ ]:
def build_kill_chain_dag(events: list) -> nx.DiGraph:
    """
    Construit un graphe DAG représentant la kill chain d'une attaque.
    Chaque noeud = un événement, chaque arc = une transition temporelle.
    """
    G = nx.DiGraph()

    for i, event in enumerate(events):
        node_id = f"evt_{i}"
        G.add_node(node_id,
            label       = event['label'],
            source_ip   = event['source_ip'],
            timestamp   = event['timestamp'],
            confidence  = event['confidence'],
            att_ck      = event.get('att_ck_technique_id', 'N/A'),
            criticality = event.get('criticality_score', 0)
        )
        # Relier chaque événement au suivant
        if i > 0:
            prev_id = f"evt_{i-1}"
            G.add_edge(prev_id, node_id,
                time_delta = f"{i*15} min"
            )

    return G

# Événements d'une attaque simulée
attack_events = [
    {'label': 'PORT_SCAN',   'source_ip': '192.168.1.45', 'timestamp': '14:00', 'confidence': 0.91, 'att_ck_technique_id': 'T1046', 'criticality_score': 5},
    {'label': 'BRUTE_FORCE', 'source_ip': '192.168.1.45', 'timestamp': '14:15', 'confidence': 0.94, 'att_ck_technique_id': 'T1110', 'criticality_score': 6},
    {'label': 'WEB_ATTACK',  'source_ip': '192.168.1.45', 'timestamp': '14:30', 'confidence': 0.76, 'att_ck_technique_id': 'T1190', 'criticality_score': 7},
    {'label': 'BOT',         'source_ip': '192.168.1.45', 'timestamp': '15:00', 'confidence': 0.95, 'att_ck_technique_id': 'T1071', 'criticality_score': 10},
]

G = build_kill_chain_dag(attack_events)
print(f"Noeuds : {G.number_of_nodes()}")
print(f"Arcs   : {G.number_of_edges()}")
print(f"DAG valide : {nx.is_directed_acyclic_graph(G)}")

In [ ]:
# Couleurs selon criticité
color_map = {
    'PORT_SCAN':   '#00d4ff',
    'BRUTE_FORCE': '#ffd600',
    'WEB_ATTACK':  '#ff8800',
    'BOT':         '#ff3366'
}

node_colors = [color_map.get(G.nodes[n]['label'], '#888888') for n in G.nodes]
node_labels = {n: f"{G.nodes[n]['label']}\n{G.nodes[n]['timestamp']}\nScore:{G.nodes[n]['criticality']}" for n in G.nodes}

plt.figure(figsize=(14, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos,
    labels      = node_labels,
    node_color  = node_colors,
    node_size   = 3000,
    font_size   = 8,
    font_weight = 'bold',
    arrows      = True,
    arrowsize   = 20,
    edge_color  = '#666666',
    width       = 2
)

edge_labels = nx.get_edge_attributes(G, 'time_delta')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

plt.title("Kill Chain Reconstructor — Graphe DAG", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/kill_chain_dag.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graphe sauvegardé : data/kill_chain_dag.png")

In [ ]:
reconstructor_code = '''import networkx as nx
import json

def build_kill_chain_dag(events: list) -> nx.DiGraph:
    G = nx.DiGraph()
    for i, event in enumerate(events):
        node_id = f"evt_{i}"
        G.add_node(node_id,
            label       = event["label"],
            source_ip   = event["source_ip"],
            timestamp   = event["timestamp"],
            confidence  = event["confidence"],
            att_ck      = event.get("att_ck_technique_id", "N/A"),
            criticality = event.get("criticality_score", 0)
        )
        if i > 0:
            G.add_edge(f"evt_{i-1}", node_id)
    return G

def dag_to_json(G: nx.DiGraph) -> dict:
    """Convertit le DAG en JSON pour P4 (dashboard)."""
    return {
        "nodes": [{"id": n, **G.nodes[n]} for n in G.nodes],
        "edges": [{"from": u, "to": v} for u, v in G.edges],
        "is_valid_dag": nx.is_directed_acyclic_graph(G),
        "total_events": G.number_of_nodes(),
        "attack_duration": f"{G.number_of_nodes() * 15} min"
    }

if __name__ == "__main__":
    events = [
        {"label": "PORT_SCAN",   "source_ip": "1.2.3.4", "timestamp": "14:00", "confidence": 0.91, "att_ck_technique_id": "T1046", "criticality_score": 5},
        {"label": "BRUTE_FORCE", "source_ip": "1.2.3.4", "timestamp": "14:15", "confidence": 0.94, "att_ck_technique_id": "T1110", "criticality_score": 6},
        {"label": "BOT",         "source_ip": "1.2.3.4", "timestamp": "15:00", "confidence": 0.95, "att_ck_technique_id": "T1071", "criticality_score": 10},
    ]
    G    = build_kill_chain_dag(events)
    data = dag_to_json(G)
    print(json.dumps(data, indent=2))
'''

with open('../src/kill_chain_reconstructor.py', 'w', encoding='utf-8') as f:
    f.write(reconstructor_code)
print("✅ kill_chain_reconstructor.py sauvegardé !")